# Comparar con el mercado directo

Primero se mide el valor añadido de usar cuotas. Después se compara la señal deportiva con una referencia que no utiliza aprendizaje.

## Inicialización

Usamos la misma identidad de datos, código y protocolo en todos los cuadernos.

In [1]:
from pathlib import Path
import sys, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd()
while not (ROOT/'scripts/mlcore').is_dir():
    if ROOT == ROOT.parent: raise RuntimeError('No se encontro la raiz del proyecto')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'scripts'))
from mlcore.experiment import initialize, aligned
from mlcore.data import read_data, FOLDS, TEST
OUT = initialize()
print('Ejecucion verificada:', OUT.name)


Ejecucion verificada: 01afe756130bafc6


**Qué encontramos y cómo se usa.** La identidad evita mezclar artefactos de protocolos distintos. Los cálculos científicos viven en `scripts/mlcore`; estos cuadernos explican, ejecutan y muestran cada etapa.

## Referencias observadas

Consenso y Pinnacle con respaldo son distribuciones directas. Las casas incompletas o no fiables no se imputan.

In [2]:
from mlcore.experiment import references
from mlcore.metrics import metrics
df, sets = read_data(OUT)
rows=[]
for s in FOLDS:
    va=df[df.Season.eq(s)]; tr=df[df.Season.lt(s)]
    for name,p in references(va,tr).items(): rows.append(dict(Season=s,referencia=name,**metrics(va.y,p)))
references_table=pd.DataFrame(rows)
display(references_table.groupby('referencia')[['log_loss','AUC_macro','ECE_masa']].mean())

/home/evssz/UTEC/6to/machine/ML_Project/scripts/mlcore/data.py:290: DtypeWarning: Columns (0: Time, 1: B365CH) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(out/'dataset.csv', parse_dates=['Date', 'H_snapshot_at', 'A_snapshot_at'])


,log_loss,AUC_macro,ECE_masa
referencia,,,
Pinnacle_respaldo,0.965631,0.684448,0.022221
consenso,0.966375,0.684013,0.022871
prior,1.071773,0.500000,0.013165


**Qué encontramos / decisión.** Las medias por temporada son un diagnóstico; la tabla principal final agregará por partido. El prior se estima con entrenamiento.

## Diagnósticos pareados e importancia

Calculamos comparaciones descriptivas de validación y permutación por grupos en LR. No se usan para mirar el test.

In [3]:
from mlcore.report import validation_diagnostics
if not (OUT/'validation_diagnostics.csv').exists(): validation_diagnostics(OUT)
display(pd.read_csv(OUT/'validation_diagnostics.csv'))
display(pd.read_csv(OUT/'importance_groups.csv').groupby(['branch','grupo']).delta.agg(['mean','std']))

,candidate,n,log_loss,RPS,Brier,AUC_macro,accuracy,balanced_accuracy,ECE_masa,ECE_ancho,acierto_ganador_condicional,recall_empate,delta,lo,hi,p,temporadas_mejora,p_BH
0,con/LR,18184,0.969894,0.196020,0.575624,0.679279,0.535691,0.464694,0.008755,0.007535,0.712258,0.020137,0.003460,0.001820,0.005154,0.000400,1/6,0.000533
1,con/HGB,18184,0.968530,0.195704,0.574816,0.680613,0.538495,0.464325,0.012702,0.009617,0.716795,0.000000,0.002095,0.000978,0.003241,0.000800,1/6,0.000960
2,con/XGB,18184,0.968308,0.195689,0.574686,0.681327,0.537890,0.464026,0.012335,0.010611,0.715843,0.000664,0.001873,0.000575,0.003173,0.005599,2/6,0.006108
3,con/RF,18184,0.967576,0.195493,0.574173,0.681364,0.537615,0.464797,0.008830,0.007731,0.715185,0.005975,0.001142,-0.000264,0.002509,0.101980,3/6,0.101980
4,con/MLP,18184,0.972342,0.197120,0.577942,0.675483,0.530576,0.457067,0.011157,0.010081,0.706550,0.006639,0.005907,0.004015,0.007886,0.000400,1/6,0.000533
5,con/SVM,18184,0.994520,0.203678,0.592727,0.654663,0.522657,0.444055,0.022765,0.020354,0.695499,0.000000,0.028085,0.024889,0.031277,0.000400,0/6,0.000533
6,sin/LR,18184,0.980137,0.199215,0.582520,0.669522,0.529037,0.453824,0.016032,0.015056,0.704135,0.000885,0.013703,0.011425,0.016058,0.000400,0/6,0.000533
7,sin/HGB,18184,0.979474,0.199331,0.582538,0.669827,0.528982,0.451883,0.013270,0.011538,0.703915,0.000000,0.013039,0.010577,0.015501,0.000400,0/6,0.000533
8,sin/XGB,18184,0.978682,0.199048,0.581911,0.670019,0.529476,0.452574,0.013702,0.013340,0.704866,0.000000,0.012247,0.009819,0.014677,0.000400,0/6,0.000533
9,sin/RF,18184,0.979361,0.199225,0.582278,0.669085,0.529531,0.452763,0.012624,0.012710,0.704720,0.000443,0.012926,0.010352,0.015427,0.000400,0/6,0.000533


mean       std
branch grupo                          
con    Habilidades  0.008234  0.006293
       Historia     0.009571  0.010581
       Mercado      0.169259  0.108140
       Plantilla    0.001059  0.000633
sin    Habilidades  0.022066  0.005423
       Historia     0.081205  0.017703
       Plantilla    0.001993  0.001206

**Qué encontramos / decisión.** Estos contrastes diagnósticos corrigen su propia familia; las cinco comparaciones principales finales se declararán aparte. Importancia por permutación no equivale a causalidad.